In [13]:

#1. Install the medmnist Library
%pip install medmnist
#2. Python Commands to Load the Dataset
import medmnist
from medmnist import PneumoniaMNIST
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Define any transformations (e.g., convert to PyTorch tensors)
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# 1. Download and load the datasets
train_dataset = PneumoniaMNIST(split='train', transform=data_transform, download=True)
val_dataset = PneumoniaMNIST(split='val', transform=data_transform, download=True)
test_dataset = PneumoniaMNIST(split='test', transform=data_transform, download=True)

# 2. Wrap them in PyTorch DataLoaders for your model
batch_size = 128
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)


In [14]:

import torch
import torch.nn as nn #import neural network for torch
import torch.optim as optim #import optimizer for neural network

# build CNN from nn.Module
class SimpleXrayCNN(nn.Module):
    def __init__(self):

        super().__init__() # initialize
        #  first feature scan
        # Conv2d creates sliding window of 3x3 pixels that scan the image for basic patterns like edgess. It uses 1 channel as images are greyscale and outputs 16 feature maps.
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        # ReLU function (Rectified Linear Unit) turns negative numbers to 0.
        self.relu1 = nn.ReLU()
        # maxPool shrinks the image size from 28x28 down to 14x14 to focus on the strongest features
        self.pool1 = nn.MaxPool2d(2, 2)


        # run a second feature scan
        # takes the 16 outputs from the last layer and expands to 32.
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        # shrinks the image from 14x14 down to 7x7
        self.pool2 = nn.MaxPool2d(2, 2)


        # linear layers need a flat list of numbers, not a 2D image grid.
        # final image size is 7x7 with 32 channels, (32)(7)(7) = 1568 total points.
        self.fc1 = nn.Linear(1568, 64) # funnels the 1568 points down to 64.
        self.relu3 = nn.ReLU()

        # final output layer. funnels the 64 down to just 1 number.
        # positve is for pneumonia and negative is for normal label
        self.fc2 = nn.Linear(64, 1)

    # The forward function defines the path the image takes through the layers
    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))

        # .view() function to flatten the 2D grid into the 1D list needed for linear layers.
        x = x.view(-1, 1568)

        x = self.relu3(self.fc1(x))
        x = self.fc2(x) # The final raw score
        return x

# use Colab's GPU
device = torch.device('cuda')
# Create omodel and immediately pass to GPU
model = SimpleXrayCNN().to(device)

# The loss function calculates the error for each round
loss_fn = nn.BCEWithLogitsLoss()

# optimizer adjusts the weights based on error rate
# lr controls how big of a step the optimizer takes per adjustment.
optimizer = optim.Adam(model.parameters(), lr=0.001)


# function to evaulate model performance
def evaluate(model, dataloader):
    # .eval() locks the model's weights
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in dataloader:
            # Move data to the GPU
            images, labels = images.to(device), labels.to(device).float()

            # Ask the model for its predictions
            preds = model(images)
            # Calculate the error
            loss = loss_fn(preds, labels)
            total_loss += loss.item() * images.size(0)

            # If the raw score is >= 0, it predicted class 1 (Pneumonia).
            binary_preds = (preds >= 0.0).float()
            # Count how many predictions exactly matched the real labels
            correct += (binary_preds == labels).sum().item()
            total += labels.size(0)

    # Return average error and accuracy percentage
    return total_loss / total, correct / total





In [15]:
#train model with 5 epochs using only training data
num_epochs = 5

for epoch in range(num_epochs):
    # set model to model.train() so it can learn
    model.train()
    running_loss = 0.0

    # loop through the training data one batch at a time, passing both labels and images
    for images, labels in train_loader:

        images, labels = images.to(device), labels.to(device).float()

        # Wipe the calculated gradients from the previous batch
        optimizer.zero_grad()

        # forward pass to predicate see what the model guesses
        preds = model(images)

        # determine error
        loss = loss_fn(preds, labels)

        # Backward pass to identify which weights to adjust
        loss.backward()

        # optimize weights
        optimizer.step()

        # update running tally of the error
        running_loss += loss.item() * images.size(0)

    # average error for this epoch
    train_loss = running_loss / len(train_dataset)

    # Run our evaluate function on the validation data to see if it's actually learning.
    val_loss, val_acc = evaluate(model, val_loader)

    # print results for this epoch
    print(f"epoch [{epoch+1}/{num_epochs}] train_loss: {train_loss}, val_loss: {val_loss}, val_acc: {val_acc*100}%")



Epoch [1/5] -> Train Loss: 0.5167165208440163 | Val Loss: 0.36593669903187354 | Val Acc: 82.25190839694656%
Epoch [2/5] -> Train Loss: 0.21941121193495663 | Val Loss: 0.18909865714439 | Val Acc: 91.79389312977099%
Epoch [3/5] -> Train Loss: 0.1565657367165451 | Val Loss: 0.14601683946511218 | Val Acc: 94.46564885496184%
Epoch [4/5] -> Train Loss: 0.13778920731204186 | Val Loss: 0.12731335969030402 | Val Acc: 94.8473282442748%
Epoch [5/5] -> Train Loss: 0.1285575063505108 | Val Loss: 0.12077352148658446 | Val Acc: 95.61068702290076%


In [16]:
# Use model on held out test data to determine actual performance
test_loss, test_acc = evaluate(model, test_loader)
print(f"Test data score: {test_acc*100}%")

Test data score: 82.8525641025641%
